# Business Analyst Intern Technical Assessment
## Section 1 - Data Quality Checks

This notebook solves **only Section 1** using Python and Pandas.

### 1. Load Dataset

Load the CSV and preserve the loyalty tier value "None" as a valid category.

In [1]:
import pandas as pd

csv_path = "hotel_bookings (1).csv"
df = pd.read_csv(csv_path, keep_default_na=False)
rows_before = len(df)

print("Number of rows loaded:", rows_before)
df.head()

Number of rows loaded: 12000


,booking_id,customer_id,customer_name,customer_segment,customer_signup_date,customer_home_city,customer_loyalty_tier,property_id,property_name,property_city,...,nights,booking_channel,adr,discount_amount,coupon_code,total_amount,payment_method,booking_status,review_rating,review_date
0,100000,424,Customer_424,Group,2023-07-03,Manali,Platinum,38,Crimson Courtyard,Manali,...,4,OTA,5808.46,0.00,NONE,46467.68,Debit Card,Cancelled,,
1,100001,239,Customer_239,Individual,2022-07-18,Jaipur,None,32,Saffron Palace,Pune,...,4,Corporate Portal,4021.62,0.00,,16086.48,Net Banking,Cancelled,,
2,100002,301,Customer_301,Corporate,2023-07-05,Jaipur,Gold,53,Saffron Heights,Chennai,...,3,Corporate Portal,17663.11,15373.35,SAVE10,90605.31,Net Banking,Completed,,
3,100003,722,Customer_722,Individual,2022-11-07,Udaipur,None,43,Indigo Lodge,Bangalore,...,3,OTA,5885.85,0.00,NONE,17657.55,Credit Card,No-Show,,
4,100004,306,Customer_306,Corporate,2022-02-02,Udaipur,Silver,29,Cedar Lodge,Kochi,...,7,Direct Website,6199.44,5924.82,SAVE10,80867.34,Credit Card,Completed,6.0,2024-11-21


### 2. Convert date columns
Convert date fields into datetime format and convert review ratings to numeric values.

In [2]:
# Converting the date columns to datetime
df["customer_signup_date"] = pd.to_datetime(df["customer_signup_date"], errors="coerce")
df["booking_date"] = pd.to_datetime(df["booking_date"], errors="coerce")
df["checkin_date"] = pd.to_datetime(df["checkin_date"], errors="coerce")
df["checkout_date"] = pd.to_datetime(df["checkout_date"], errors="coerce")
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")

# Converting review ratings to numeric values
df["review_rating"] = pd.to_numeric(df["review_rating"], errors="coerce")

### 3. Create Data Quality flags
Create flags for invalid stays, bookings before signup, zero-room bookings, and cancelled bookings with reviews.

In [3]:
# Footnote 1: checkout must be after check-in
df["invalid_stay"] = df["checkout_date"] <= df["checkin_date"]

# Footnote 2: booking should not occur before customer signup
df["booking_before_signup"] = df["booking_date"] < df["customer_signup_date"]

# Footnote 3: a booking must contain at least one room
df["zero_rooms"] = df["num_rooms"] == 0

# Footnote 5: cancelled bookings cannot have reviews
df["cancelled_with_review"] = (
    (df["booking_status"] == "Cancelled")
    & df["review_rating"].notna()
)

# Displaying the number of rows affected by each issue
quality_issue_counts = pd.Series({
    "Invalid stays": int(df["invalid_stay"].sum()),
    "Bookings before signup": int(df["booking_before_signup"].sum()),
    "Zero-room bookings": int(df["zero_rooms"].sum()),
    "Cancelled bookings with reviews": int(df["cancelled_with_review"].sum())
})

quality_issue_counts

Invalid stays                      120
Bookings before signup             163
Zero-room bookings                  60
Cancelled bookings with reviews     50
dtype: int64

## A1. Invalid Stay Count
Count bookings where checkout_date is on or before checkin_date.

In [4]:
# Counting invalid stays
invalid_stay_count = int(df["invalid_stay"].sum())

print("A1 - Number of invalid stays:", invalid_stay_count)

A1 - Number of invalid stays: 120


## A2. Review Rating Analysis by Customer Segment

To compare review ratings, I removed clearly invalid booking records and excluded reviews attached to cancelled bookings.

Ratings are then summarized separately for each customer segment.

In [5]:
# Keeping rows that do not have a row-level booking error
valid_booking_rows = (
    ~df["invalid_stay"]
    & ~df["booking_before_signup"]
    & ~df["zero_rooms"]
)

# Keeping only possible, non-missing reviews
valid_review_rows = (
    valid_booking_rows
    & ~df["cancelled_with_review"]
    & df["review_rating"].notna()
)

# Calculating the review statistics separately for each segment
review_summary = (
    df.loc[valid_review_rows]
      .groupby("customer_segment")["review_rating"]
      .agg(["min", "max", "mean"])
      .round(2)
)

review_summary

,min,max,mean
customer_segment,,,
Corporate,3.0,10.0,7.23
Group,1.0,5.0,3.76
Individual,1.0,5.0,3.77


Corporate ratings are on a 1-10 scale, while Individual and Group ratings are on a 1-5 scale, so raw averages must not be compared across segments without normalization.

## A3. Realized Revenue for Luxury Properties

Only completed bookings contribute to realized revenue. Invalid stays, booking-before-signup records, and zero-room bookings are excluded.

In [6]:
# Selecting valid, completed bookings for Luxury properties
luxury_completed_rows = (
    valid_booking_rows
    & (df["property_type"] == "Luxury")
    & (df["booking_status"] == "Completed")
)

# Sum total_amount to get realized revenue
luxury_realized_revenue = df.loc[
    luxury_completed_rows, "total_amount"
].sum()

print(f"A3 - Luxury realized revenue: ₹{luxury_realized_revenue:,.2f}")

A3 - Luxury realized revenue: ₹89,009,636.79


## Create Clean Dataset for Later Analysis

Create a cleaned version of the dataset by removing invalid stays, booking-before-signup records, and zero-room bookings.

Cancelled bookings are retained because they may be useful for later cancellation analysis.

In [7]:
# Removing row-level booking errors and make an independent copy
df_clean = df.loc[valid_booking_rows].copy()

# Clearing impossible review information on cancelled bookings
cancelled_review_rows_clean = (
    (df_clean["booking_status"] == "Cancelled")
    & df_clean["review_rating"].notna()
)

df_clean.loc[cancelled_review_rows_clean, "review_rating"] = pd.NA
df_clean.loc[cancelled_review_rows_clean, "review_date"] = pd.NaT

# Reset the row index after cleaning
df_clean = df_clean.reset_index(drop=True)

# Store the final row count
rows_after = len(df_clean)

print("Number of rows before cleaning:", rows_before)
print("Number of rows after cleaning:", rows_after)
print("Number of unique rows removed:", rows_before - rows_after)

Number of rows before cleaning: 12000
Number of rows after cleaning: 11661
Number of unique rows removed: 339


### Cleaning Summary

Summarize the cleaning steps and the number of affected records.

In [8]:
# Show the cleaning steps and their effects
cleaning_summary = pd.DataFrame({
    "Cleaning step": [
        "Removed invalid stays",
        "Removed bookings before customer signup",
        "Removed zero-room bookings",
        "Cleared reviews from cancelled bookings",
        "Preserved loyalty tier string 'None'",
        "Retained only Completed bookings for realized-revenue calculations"
    ],
    "Rows/values affected": [
        int(df["invalid_stay"].sum()),
        int(df["booking_before_signup"].sum()),
        int(df["zero_rooms"].sum()),
        int(df["cancelled_with_review"].sum()),
        int((df["customer_loyalty_tier"] == "None").sum()),
        int((df["booking_status"] == "Completed").sum())
    ],
    "Action": [
        "Rows removed from df_clean",
        "Rows removed from df_clean",
        "Rows removed from df_clean",
        "Review rating and review date cleared; booking retained if otherwise valid",
        "No removal; valid category retained",
        "Filter used for revenue only; other statuses remain in df_clean"
    ]
})

cleaning_summary

,Cleaning step,Rows/values affected,Action
0,Removed invalid stays,120,Rows removed from df_clean
1,Removed bookings before customer signup,163,Rows removed from df_clean
2,Removed zero-room bookings,60,Rows removed from df_clean
3,Cleared reviews from cancelled bookings,50,Review rating and review date cleared; booking...
4,Preserved loyalty tier string 'None',5571,No removal; valid category retained
5,Retained only Completed bookings for realized-...,9333,Filter used for revenue only; other statuses r...


## Key Findings

### A1
120 bookings have invalid stay dates.

### A2
Corporate customers use a 1–10 review scale, while Individual and Group customers use a 1–5 scale. Review averages should therefore not be compared directly across segments.

### A3
Realized revenue for Luxury properties after applying the required data-quality filters is ₹89,009,636.79.

### Cleaning Result
Rows before cleaning: 12,000

Rows after cleaning: 11,661

Rows removed: 339